# Pruning

In [97]:
import torch
from torch import nn
import torch_pruning as tp
from transformers import GPT2LMHeadModel, GPT2Tokenizer

## Upload GPT-2 model

In [98]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [99]:
def inference(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.8
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [100]:
def describe_model(model):
    for i, block in enumerate(model.transformer.h):
        print(
            f"Block {i}:",
            f"c_fc={tuple(block.mlp.c_fc.weight.shape)},",
            f"c_proj={tuple(block.mlp.c_proj.weight.shape)}",
        )

In [101]:
tokenized_prompt, answer = inference(model, prompt="Hello, how are you?")
print(answer)

Hello, how are you?

I've been doing this for five days and it's been really great. I've had a lot of love from my fans and a lot of great support from all the people who support me.

As well as being able to have a lot of sleep, I have the ability to get up early so I can have more time to enjoy myself and not work and not have to go to the studio.

I've been looking all over the place and I'm really happy to


## Model mapping

In [102]:
from transformers.pytorch_utils import Conv1D


def convert_model_to_torch(module):
    for name, child in list(module.named_children()):
        if isinstance(child, Conv1D):
            in_features, out_features = child.weight.shape

            linear = nn.Linear(
                in_features=in_features,
                out_features=out_features,
                bias=child.bias is not None,
                device=child.weight.device,
                dtype=child.weight.dtype,
            )

            with torch.no_grad():
                linear.weight.copy_(child.weight.T)

                if child.bias is not None:
                    linear.bias.copy_(child.bias)

            setattr(module, name, linear)
        else:
            convert_model_to_torch(child)

    return module

In [103]:
model = convert_model_to_torch(model)

## Structure

In [104]:
describe_model(model)

Block 0: c_fc=(3072, 768), c_proj=(768, 3072)
Block 1: c_fc=(3072, 768), c_proj=(768, 3072)
Block 2: c_fc=(3072, 768), c_proj=(768, 3072)
Block 3: c_fc=(3072, 768), c_proj=(768, 3072)
Block 4: c_fc=(3072, 768), c_proj=(768, 3072)
Block 5: c_fc=(3072, 768), c_proj=(768, 3072)
Block 6: c_fc=(3072, 768), c_proj=(768, 3072)
Block 7: c_fc=(3072, 768), c_proj=(768, 3072)
Block 8: c_fc=(3072, 768), c_proj=(768, 3072)
Block 9: c_fc=(3072, 768), c_proj=(768, 3072)
Block 10: c_fc=(3072, 768), c_proj=(768, 3072)
Block 11: c_fc=(3072, 768), c_proj=(768, 3072)


## Model before pruning

In [105]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 124439808


## Pruning experiments

In [128]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_blocks(model, ratio=0.25):
    model = deepcopy(model)

    importance = tp.importance.GroupMagnitudeImportance(p=2)

    mlp_fc_layers = {
        block.mlp.c_fc for block in model.transformer.h
    }

    ignored_layers = [
        module for module in model.modules() if isinstance(module, nn.Linear) and module not in mlp_fc_layers
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"],
        forward_fn=forward_fn,
        importance=importance,
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [129]:
pruned_model = prune_blocks(model, ratio=0.1)

In [130]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? And that's when you make that very obvious, when you make that really explicit on your own. And just on your own in your own way, and in your own very personal way, in your own way, in your own way, and in your own way, and where you're going, and where you're going, and where you're in your own way, and there's going to be what's going on, where you're moving, what's going to be coming to me,


In [131]:
describe_model(pruned_model)

Block 0: c_fc=(2760, 768), c_proj=(768, 2760)
Block 1: c_fc=(2760, 768), c_proj=(768, 2760)
Block 2: c_fc=(2760, 768), c_proj=(768, 2760)
Block 3: c_fc=(2760, 768), c_proj=(768, 2760)
Block 4: c_fc=(2760, 768), c_proj=(768, 2760)
Block 5: c_fc=(2760, 768), c_proj=(768, 2760)
Block 6: c_fc=(2760, 768), c_proj=(768, 2760)
Block 7: c_fc=(2760, 768), c_proj=(768, 2760)
Block 8: c_fc=(2760, 768), c_proj=(768, 2760)
Block 9: c_fc=(2760, 768), c_proj=(768, 2760)
Block 10: c_fc=(2760, 768), c_proj=(768, 2760)
Block 11: c_fc=(2760, 768), c_proj=(768, 2760)


In [132]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 118685280


## Simple test (before vs after)

In [133]:
def evaluate_model(model, text, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            labels=inputs["input_ids"],
            use_cache=False,
        )

    loss = outputs.loss.item()

    return { "loss": loss }

In [134]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model, test_text, tokenizer))

Original: {'loss': 4.7835798263549805}
Pruned:   {'loss': 5.129265785217285}
